In [8]:
%load_ext autoreload
%autoreload 2

import sys

sys.path.append("..")
import pandas as pd
import numpy as np
from src import metrics
import yfinance as yf

df = yf.download('SPY', start='2010-01-01', auto_adjust=True, multi_level_index=False)

df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2010-01-04,84.368980,84.413646,83.014074,83.654305,118944600
2010-01-05,84.592331,84.629556,84.011658,84.316887,111579900
2010-01-06,84.651878,84.860325,84.443432,84.510430,116074400
2010-01-07,85.009201,85.113424,84.257301,84.495526,131091100
2010-01-08,85.292107,85.329332,84.614656,84.785878,126402800


In [9]:
close = df['Close']

ret = metrics.daily_returns(close)

y = metrics.rolling_vol(ret, window=20).shift(-20).dropna()
log_y = np.log(y)

print(y)

Date
2010-01-04    0.168030
2010-01-05    0.167857
2010-01-06    0.196891
2010-01-07    0.195957
2010-01-08    0.194775
                ...   
2026-08-14    0.088279
2026-08-17    0.088202
2026-08-18    0.086543
2026-08-19    0.096396
2026-08-20    0.091761
Name: Close, Length: 4183, dtype: float64


In [10]:
vol5 = metrics.rolling_vol(ret, window=5)
vol20 = metrics.rolling_vol(ret, window=20)
vol60 = metrics.rolling_vol(ret, window=60)

ewm_vol = ret.ewm(span=20).std() * np.sqrt(252)
vol_of_vol = vol20.rolling(20).std()

abs_ret = ret.abs()

neg_flag = (ret < 0).astype(int)

hl_range = (df['High'] - df['Low']) / df['Close']

vol_ratio = df['Volume'] / df['Volume'].rolling(20).mean()

X = pd.DataFrame({
    'vol5': vol5,
    'vol20': vol20,
    'vol60': vol60,
    'ewm_vol': ewm_vol,
    'vol_of_vol': vol_of_vol,
    'ret' : ret,
    'abs_ret': abs_ret,
    'neg_flag': neg_flag,
    'hl_range': hl_range,
    'vol_ratio': vol_ratio
})

In [11]:
data = pd.concat([X, log_y.rename('target')], axis=1).dropna()

X = data.drop('target', axis=1)
y_log = data['target']

print(X.shape, y_log.shape)
print(X.index[0], X.index[-1])
X.describe().T.round(3)



(4123, 10) (4123,)
2010-03-31 00:00:00 2026-08-20 00:00:00


,count,mean,std,min,25%,50%,75%,max
vol5,4123.0,0.137,0.107,0.007,0.072,0.113,0.171,1.402
vol20,4123.0,0.146,0.091,0.032,0.093,0.124,0.173,0.932
vol60,4123.0,0.152,0.080,0.050,0.108,0.129,0.170,0.614
ewm_vol,4123.0,0.148,0.089,0.035,0.096,0.124,0.173,0.941
vol_of_vol,4123.0,0.022,0.024,0.002,0.010,0.016,0.025,0.242
ret,4123.0,0.001,0.011,-0.109,-0.004,0.001,0.006,0.105
abs_ret,4123.0,0.007,0.008,0.000,0.002,0.005,0.010,0.109
neg_flag,4123.0,0.446,0.497,0.000,0.000,0.000,1.000,1.000
hl_range,4123.0,0.011,0.008,0.001,0.006,0.009,0.013,0.106
vol_ratio,4123.0,1.006,0.344,0.304,0.777,0.940,1.149,3.557


In [12]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5, gap=20)
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    print(i, X.index[train_idx[0]].date(), X.index[train_idx[-1]].date(),
          "|", X.index[test_idx[0]].date(), X.index[test_idx[-1]].date())


0 2010-03-31 2012-11-21 | 2012-12-21 2015-09-15
1 2010-03-31 2015-08-17 | 2015-09-16 2018-06-07
2 2010-03-31 2018-05-09 | 2018-06-08 2021-03-02
3 2010-03-31 2021-02-01 | 2021-03-03 2023-11-21
4 2010-03-31 2023-10-24 | 2023-11-22 2026-08-20


In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=5, gap=20)

baselines = {
    "naive":     np.log(X["vol20"]),
    "ewma":      np.log(X["ewm_vol"]),
    "roll_mean": np.log(X["vol20"].rolling(20).mean()),
}

results = []
for i, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train, y_train = X.iloc[train_idx], y_log.iloc[train_idx]
    X_test,  y_test  = X.iloc[test_idx],  y_log.iloc[test_idx]

    # baselines: no training, just score
    for name, pred in baselines.items():
        p = pred.iloc[test_idx]
        results.append({
            "fold": i, "model": name,
            "rmse": np.sqrt(mean_squared_error(y_test, p)),
            "mae":  mean_absolute_error(y_test, p),
        })

    # ridge: train on this fold's past, predict its future
    model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
    model.fit(X_train, y_train)
    p = model.predict(X_test)
    results.append({
        "fold": i, "model": "ridge",
        "rmse": np.sqrt(mean_squared_error(y_test, p)),
        "mae":  mean_absolute_error(y_test, p),
    })

res = pd.DataFrame(results)
tbl = res.pivot(index="fold", columns="model", values="rmse")
tbl.loc["mean"] = tbl.mean()
tbl.round(3)

model,ewma,naive,ridge,roll_mean
fold,,,,
0,0.442,0.491,0.383,0.435
1,0.471,0.488,0.433,0.496
2,0.564,0.589,0.517,0.653
3,0.319,0.342,0.298,0.322
4,0.445,0.455,0.372,0.476
mean,0.448,0.473,0.401,0.476


In [14]:
coefs = pd.Series(model[-1].coef_, index=X.columns).sort_values()
coefs.round(3)

vol20        -0.155
vol_of_vol   -0.111
abs_ret      -0.040
ret          -0.018
vol5         -0.014
neg_flag      0.021
vol_ratio     0.031
vol60         0.083
hl_range      0.113
ewm_vol       0.408
dtype: float64